# Clean up of redundant rows and columns in the merged Booking and Measurements Dataset

---

In [1]:
import duckdb

In [2]:
# Paths
merged_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements.parquet"
cleaned_output_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements_cleaned.parquet"
balanced_output_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_cleaned_balanced_15k.parquet"

In [3]:
# Connect to DuckDB
con = duckdb.connect()

In [4]:
# Load merged dataset as a view
con.execute(f"CREATE OR REPLACE VIEW merged AS SELECT * FROM '{merged_file}';")

In [5]:
# Drop exact duplicate rows
con.execute("CREATE OR REPLACE TEMP TABLE merged_dedup AS SELECT DISTINCT * FROM merged;")

In [6]:
# Compute similarity report for numerical/categorical columns (serial_number_id, station_id, book_state)
similarity_report = con.execute("""
    WITH stats AS (
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN serial_number_id = serial_number_id_1 THEN 1 ELSE 0 END) AS serial_match,
            SUM(CASE WHEN station_id = station_id_1 THEN 1 ELSE 0 END) AS station_match,
            SUM(CASE WHEN book_state = book_state_1 THEN 1 ELSE 0 END) AS book_state_match
        FROM merged_dedup
    )
    SELECT
        ROUND(serial_match::DOUBLE / total_rows, 4) AS serial_number_id_match_ratio,
        ROUND(station_match::DOUBLE / total_rows, 4) AS station_id_match_ratio,
        ROUND(book_state_match::DOUBLE / total_rows, 4) AS book_state_match_ratio
    FROM stats;
""").fetchdf()

print("Similarity Report (exact matches):")
print(similarity_report)

Similarity Report (exact matches):
   serial_number_id_match_ratio  station_id_match_ratio  \
0                        0.9999                  0.9999   

   book_state_match_ratio  
0                     1.0  


In [7]:
# Handle datetime approximate matches within tolerance
time_report = con.execute("""
    WITH stats AS (
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN ABS(DATE_DIFF('milliseconds', created_at, created_at_1)) <= 500 THEN 1 ELSE 0 END) AS close_match
        FROM merged_dedup
    )
    SELECT
        ROUND(close_match::DOUBLE / total_rows, 4) AS created_at_close_match_ratio,
        ROUND(1 - (close_match::DOUBLE / total_rows), 4) AS created_at_mismatch_ratio
    FROM stats;
""").fetchdf()

print("Datetime Similarity Report (within ±500ms):")
print(time_report)

Datetime Similarity Report (within ±500ms):
   created_at_close_match_ratio  created_at_mismatch_ratio
0                        0.9969                     0.0031


In [8]:
# Create cleaned table:
con.execute("""
    CREATE OR REPLACE TABLE merged_cleaned AS
    SELECT
        -- Use COALESCE for categorical columns (prefer the left unless NULL)
        COALESCE(serial_number_id, serial_number_id_1) AS serial_number_id,
        COALESCE(station_id, station_id_1) AS station_id,
        COALESCE(book_state, book_state_1) AS book_state,
        -- For datetime, choose created_at if within 500ms, otherwise use created_at_1
        CASE
            WHEN ABS(DATE_DIFF('milliseconds', created_at, created_at_1)) <= 500 THEN created_at
            ELSE created_at_1
        END AS created_at,
        -- Include remaining non-duplicate columns
        *
        EXCLUDE(serial_number_id, serial_number_id_1, station_id, station_id_1, book_state, book_state_1, created_at, created_at_1)
    FROM merged_dedup;
""")

In [9]:
con.execute("""
    CREATE OR REPLACE TABLE merged_cleaned_balanced AS
    (
        SELECT * FROM merged_cleaned
        WHERE book_state = 1
        ORDER BY RANDOM()
        LIMIT 15000
    )
    UNION ALL
    (
        SELECT * FROM merged_cleaned
        WHERE book_state = 0
        ORDER BY RANDOM()
        LIMIT 15000
    );
""")

In [10]:
# Save to Parquet
con.execute(f"""
    COPY merged_cleaned_balanced TO '{balanced_output_file}' (FORMAT 'parquet', COMPRESSION 'zstd');
""")

counts = con.execute("""
    SELECT book_state, COUNT(*) AS count
    FROM merged_cleaned_balanced
    GROUP BY book_state
    ORDER BY book_state;
""").fetchdf()

print(f"\nBalanced dataset saved to: {balanced_output_file}")
print("\nBalanced book_state counts:")
print(counts)


Balanced dataset saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_cleaned_balanced_15k.parquet

Balanced book_state counts:
   book_state  count
0           0  15000
1           1  15000


In [11]:
# Final summary
shape = con.execute("SELECT COUNT(*) AS rows FROM merged_cleaned_balanced;").fetchdf()
print(f"\nCleaned merged file saved to: {cleaned_output_file}")
print(f"Final shape:", shape['rows'][0])

# Print head
head = con.execute("SELECT * FROM merged_cleaned_balanced LIMIT 10;").df()
print("\nHead of cleaned dataset:\n", head)


Cleaned merged file saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements_cleaned.parquet
Final shape: 30000

Head of cleaned dataset:
   serial_number_id station_id  book_state                       created_at  \
0         0731fcba   1405e64b           1 2025-05-09 10:05:37.048000+02:00   
1         7bd8b201   0c3fb4ee           1 2025-03-05 23:20:19.581000+01:00   
2         f5e9d851   38b291ac           1 2025-04-01 08:58:41.496000+02:00   
3         557a7bee   464416bb           1 2025-05-07 00:11:31.486000+02:00   
4         9153f342   38b291ac           1 2025-03-12 04:59:58.141000+01:00   
5         b3a0d299   38b291ac           1 2025-03-12 05:27:22.947000+01:00   
6         0f3779ea   1405e64b           1 2025-05-09 07:08:58.868000+02:00   
7         50f779d7   1405e64b           1 2025-05-08 21:04:36.036000+02:00   
8         84eda1b8   38b291ac           1 2025-05-08 03:44:53.980000+02:00   
9        

In [12]:
con.close()